In [ ]:
!pip install torch torchvision torchaudio
!pip install pandas numpy matplotlib tqdm tensorboard

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    
    # Google Driveをマウント
    drive.mount('/content/drive')
    
    # プロジェクトディレクトリに移動
    PROJECT_ROOT = '/content/drive/MyDrive/Visuable_for_you_tabletennis'
    os.chdir(PROJECT_ROOT)
    
    # notebooksディレクトリをパスに追加
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts/notebooks'))
    
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True
    
except ImportError:
    # ローカル環境の場合
    IN_COLAB = False
    # notebooksディレクトリ（このノートブックの場所）をパスに追加
    notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import json
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# 新しいパイプラインをインポート
from src.pipelines import TrainingPipeline, TrainingPipelineConfig
from src.pipelines import ModelConfig, DatasetConfig, OptimizerConfig, TrainingConfig

print("インポート完了")
print(f"PyTorchバージョン: {torch.__version__}")
print(f"CUDAが利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# データディレクトリのパス設定
data_root = '/content/drive/MyDrive/Visuable_for_you_tabletennis/data/detect'

# 訓練用動画のディレクトリリスト
train_dirs = [
    f"{data_root}/sample_video_03_short",
    f"{data_root}/sample_video_04_short",
    f"{data_root}/sample_video_05_02",
]

# 検証用動画のディレクトリリスト
val_dirs = [
    f"{data_root}/sample_video_06_01",
]

# 出力ディレクトリ
OUTPUT_DIR = "output/training"

# ディレクトリの存在確認
print("=" * 60)
print("データディレクトリの確認")
print("=" * 60)

import os
print("訓練用データ:")
for i, data_dir in enumerate(train_dirs, 1):
    exists = os.path.exists(data_dir)
    csv_exists = os.path.exists(os.path.join(data_dir, 'original_pose_data.csv'))
    label_exists = os.path.exists(os.path.join(data_dir, 'play_labels.csv'))
    print(f"  [{i}] {os.path.basename(data_dir)}: {'✓' if exists else '✗'}")
    if exists:
        print(f"      - pose_data.csv: {'✓' if csv_exists else '✗'}")
        print(f"      - play_labels.csv: {'✓' if label_exists else '✗'}")

print("\n検証用データ:")
for i, data_dir in enumerate(val_dirs, 1):
    exists = os.path.exists(data_dir)
    csv_exists = os.path.exists(os.path.join(data_dir, 'original_pose_data.csv'))
    label_exists = os.path.exists(os.path.join(data_dir, 'play_labels.csv'))
    print(f"  [{i}] {os.path.basename(data_dir)}: {'✓' if exists else '✗'}")
    if exists:
        print(f"      - pose_data.csv: {'✓' if csv_exists else '✗'}")
        print(f"      - play_labels.csv: {'✓' if label_exists else '✗'}")

print("=" * 60)
print("\n注意: 複数動画のCSVを自動的に統合して学習します")
print("各ディレクトリに original_pose_data.csv と play_labels.csv が必要です")

In [ ]:
# ========================================
# オプション1: デフォルト設定で学習（推奨）
# ========================================

# デバイス設定
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# デフォルト設定でパイプライン作成（複数CSV自動統合）
pipeline = TrainingPipeline.create_default(
    train_data_dirs=train_dirs,
    val_data_dirs=val_dirs,
    output_dir=OUTPUT_DIR,
    device=device
)

print("デフォルト設定で学習パイプラインを作成しました")
print(f"  訓練動画数: {len(train_dirs)}")
print(f"  検証動画数: {len(val_dirs)}")
print(f"  モデル: LSTM (hidden_size=128, num_layers=2)")
print(f"  シーケンス長: 30フレーム")
print(f"  バッチサイズ: 32")
print(f"  エポック数: 50")
print(f"  デバイス: {device}")

In [ ]:
# ========================================
# オプション2: カスタム設定で学習（上級者向け）
# ========================================
# 上記のcell-4の代わりにこちらを実行してカスタマイズできます

# モデル設定
model_config = ModelConfig(
    model_type='lstm',      # 'lstm' or 'cnn_lstm'
    hidden_size=256,        # より大きな隠れ層
    num_layers=3,           # より深いネットワーク
    dropout=0.4,
    use_attention=True
)

# データセット設定（複数CSV対応）
dataset_config = DatasetConfig(
    train_data_dirs=train_dirs,
    val_data_dirs=val_dirs,
    csv_filename='original_pose_data.csv',  # 拡張データなら 'augment_pose_data.csv'
    label_filename='play_labels.csv',
    sequence_length=45,     # より長いシーケンス
    stride=10,
    batch_size=16,          # より小さいバッチ
    num_workers=4
)

# 最適化器設定
optimizer_config = OptimizerConfig(
    learning_rate=5e-4,     # より小さい学習率
    weight_decay=1e-5,      # 正則化
    scheduler_patience=10
)

# 学習設定
training_config = TrainingConfig(
    epochs=100,
    save_every=20,
    device=device,
    use_tensorboard=True,
    early_stopping_patience=20  # Early Stopping有効化
)

# パイプライン設定を作成
custom_config = TrainingPipelineConfig(
    model=model_config,
    dataset=dataset_config,
    optimizer=optimizer_config,
    training=training_config,
    output_dir=OUTPUT_DIR
)

# カスタムパイプライン作成
# pipeline = TrainingPipeline(custom_config)  # このコメントを外して実行

print("カスタム設定の例を表示しました")
print("実際に使用するには最後の行のコメントを外してください")

In [ ]:
# ========================================
# 学習実行
# ========================================

# パイプラインを実行（全自動）
results = pipeline.run()

print("\n学習完了！")
print(f"  Best F1 Score: {results['best_val_f1']:.4f}")
print(f"  Best Model: {results['best_model_path']}")
print(f"  Final Model: {results['final_model_path']}")
print(f"  Training History: {results['history_path']}")

# 結果を保存（後で参照用）
output_dir = Path(results['output_dir'])
timestamp = output_dir.name

In [ ]:
# ========================================
# 学習曲線の可視化
# ========================================

# 学習履歴の読み込み
history_path = output_dir / 'training_history.json'
with open(history_path, 'r') as f:
    history = json.load(f)

# プロット
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
if history['val_loss']:
    axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', linewidth=2)
if history['val_acc']:
    axes[1].plot(history['val_acc'], label='Val', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(history['train_f1'], label='Train', linewidth=2)
if history['val_f1']:
    axes[2].plot(history['val_f1'], label='Val', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('F1 Score', fontsize=12)
axes[2].set_title('Training and Validation F1 Score', fontsize=14)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n学習曲線を保存: {output_dir / 'training_curves.png'}")

In [ ]:
# ========================================
# モデルの評価（オプション）
# ========================================

from src.models.play_classifier_lstm import PlayClassifierLSTM

# ベストモデルを読み込み
best_model_path = output_dir / 'best_model.pth'
checkpoint = torch.load(best_model_path, map_location=device)

# モデル再作成
model = PlayClassifierLSTM(
    input_size=34,
    hidden_size=128,  # 設定に合わせて調整
    num_layers=2,
    dropout=0.3,
    use_attention=True
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"ベストモデルを読み込み: {best_model_path}")
print(f"  エポック: {checkpoint['epoch']}")
print(f"  Best Val F1: {checkpoint['best_val_f1']:.4f}")

print("\n評価はpredict用のノートブックで実行してください")

In [ ]:
# ========================================
# Google Driveに保存（Colab環境の場合）
# ========================================

import shutil

if IN_COLAB:
    # Google Drive上の保存先
    SAVE_TO_DRIVE = f"/content/drive/MyDrive/trained_models/play_classifier/{timestamp}"
    
    # ディレクトリ作成
    os.makedirs(SAVE_TO_DRIVE, exist_ok=True)
    
    # ファイルをコピー
    files_to_copy = [
        'best_model.pth',
        'final_model.pth',
        'config.json',
        'training_history.json',
        'training_curves.png'
    ]
    
    for filename in files_to_copy:
        src = output_dir / filename
        if src.exists():
            dst = os.path.join(SAVE_TO_DRIVE, filename)
            shutil.copy2(src, dst)
            print(f"コピー完了: {filename} -> {dst}")
    
    print(f"\nモデルをGoogle Driveに保存: {SAVE_TO_DRIVE}")
else:
    print("ローカル環境では自動保存はスキップされます")
    print(f"出力ディレクトリ: {output_dir}")

In [ ]:
# ========================================
# まとめ
# ========================================

print("=" * 70)
print("学習パイプライン完了")
print("=" * 70)
print("\n新しいTrainingPipeline（複数CSV対応）を使用して学習を実行しました！")
print("\n主な利点:")
print("  ✓ コード量が大幅に削減（旧: 300行以上 → 新: 50行以下）")
print("  ✓ 複数動画のCSVを自動的に統合して学習")
print("  ✓ 設定ベースで簡単にカスタマイズ可能")
print("  ✓ 他のパイプライン（データエクスポート、拡張）と統一")
print("  ✓ 全自動で学習・保存・可視化")
print("\n複数CSV統合:")
print(f"  ✓ 訓練データ: {len(train_dirs)} 動画")
print(f"  ✓ 検証データ: {len(val_dirs)} 動画")
print("\n次のステップ:")
print("  1. 学習曲線を確認して過学習をチェック")
print("  2. predict用ノートブックでモデルを評価")
print("  3. 必要に応じてハイパーパラメータを調整して再学習")
print("  4. より多くの動画データを追加して精度向上")
print("=" * 70)